# Notebook 07 - Evaluasi Final Kandidat pada Dataset Tanpa DDS

**Tahap penelitian:** Evaluasi -> Selesai.

Notebook ini memilih satu kandidat dengan Average Precision (AP) validation tertinggi dari hasil Notebook 06, mengunci model dan threshold bila berlaku, lalu mengestimasi performanya pada test set satu kali. Hasil test tidak dipakai untuk mengganti kandidat atau threshold.


## 0. Setup


In [ ]:
import sys
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

_root_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _root_candidates if (p / 'configs' / 'config.yaml').exists() and (p / 'src').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Tidak menemukan root repo. Jalankan notebook dari root repo atau folder notebooks.')
sys.path.insert(0, str(ROOT))
OUT = ROOT / 'outputs' / 'results'
MODEL_DIR = ROOT / 'outputs' / 'models'
OUT.mkdir(parents=True, exist_ok=True)

from src.preprocessing import load_splits
from src.evaluation import evaluate, print_summary, save_results

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.25,
                     'axes.spines.top': False, 'axes.spines.right': False})
sns.set_theme(style='whitegrid', palette='colorblind')
print(f'Repo: {ROOT}')


## 1. Kunci Kandidat Berdasarkan Validation


In [ ]:
val_path = OUT / 'nb06_val_comparison.csv'
if not val_path.exists():
    raise FileNotFoundError(f'{val_path} tidak ada. Jalankan Notebook 06 terlebih dahulu.')

val_comparison = pd.read_csv(val_path, index_col=0)
if 'average_precision' not in val_comparison.columns or val_comparison.empty:
    raise ValueError('Tabel validation tidak memuat kandidat atau metrik average_precision.')
selected_val_name = val_comparison['average_precision'].astype(float).idxmax()

model_files = {
    'RandomForest': 'randomforest.pkl',
    'XGBoost': 'xgboost.pkl',
    'LightGBM': 'lightgbm.pkl',
    'RandomForest (Exp E: retuned)': 'randomforest_expE_retuned.pkl',
    'XGBoost (Exp A: retuned)': 'xgboost_expA_default.pkl',
    'XGBoost (Exp B: retuned+balanced)': 'xgboost_expB_balanced.pkl',
    'XGBoost (Exp F: soft-balanced)': 'xgboost_expF_softbalance.pkl',
    'LightGBM (Exp C: retuned+balanced)': 'lightgbm_expC_retuned.pkl',
}
selected_threshold = None
if selected_val_name.startswith('XGBoost (Exp F + threshold='):
    selected_model_name = 'XGBoost (Exp F: soft-balanced)'
    selected_threshold = float(json.loads((OUT / 'nb06_summary.json').read_text(encoding='utf-8'))['best_threshold_exp_f'])
else:
    selected_model_name = selected_val_name
if selected_model_name not in model_files:
    raise ValueError(f'Nama kandidat validation tidak dikenali: {selected_val_name}')
selected_model_file = model_files[selected_model_name]
selected_val_ap = float(val_comparison.loc[selected_val_name, 'average_precision'])
print(f'Kandidat terkunci berdasarkan AP validation: {selected_val_name}')
print(f'AP validation: {selected_val_ap:.4f}')
print(f'Artefak model: {selected_model_file}')
if selected_threshold is not None:
    print(f'Threshold terkunci dari validation: {selected_threshold:.4f}')


## 2. Muat Model Terkunci dan Test Set


In [ ]:
model_path = MODEL_DIR / selected_model_file
if not model_path.exists():
    raise FileNotFoundError(f'Model terpilih tidak ditemukan: {model_path}. Jalankan Notebook 04 dan 06.')
selected_model = joblib.load(model_path)

df_train, df_val, df_test, feature_cols, label_encoder = load_splits(ROOT / 'data' / 'processed_no_dds')
X_test = df_test[feature_cols].to_numpy(dtype=np.float32)
y_test = df_test['label'].to_numpy()
print(f'Test set: {X_test.shape[0]:,} baris, {len(feature_cols)} fitur noDDS')
print(f'Positif test: {y_test.mean():.2%}')

proba_test = selected_model.predict_proba(X_test)
if selected_threshold is None:
    pred_test = selected_model.predict(X_test)
else:
    pred_test = (proba_test[:, 1] >= selected_threshold).astype(int)
final_result = evaluate(y_test, pred_test, proba_test, label_encoder, model_name=selected_model_name)
print_summary(final_result)
save_results(final_result, OUT / 'nb07_selected_candidate_test.json')


## 3. Metrik Evaluasi Final


In [ ]:
metric_names = ['accuracy', 'f1_macro', 'f1_weighted', 'roc_auc', 'average_precision', 'precision_positive', 'recall_positive', 'f1_positive']
metric_rows = [{'Metrik': name, 'Nilai': final_result[name]} for name in metric_names if name in final_result]
metric_table = pd.DataFrame(metric_rows)
display(metric_table.style.format({'Nilai': '{:.4f}'}).set_caption(f'Evaluasi test final ? {selected_model_name}'))
metric_table.to_csv(OUT / 'nb07_selected_candidate_metrics.csv', index=False)


## 4. Confusion Matrix Kandidat Terkunci


In [ ]:
from sklearn.metrics import confusion_matrix

class_names = ['Tidak cocok par (0)', 'Cocok par (1)']
mat = confusion_matrix(y_test, pred_test, labels=[0, 1])
row_totals = mat.sum(axis=1, keepdims=True)
mat_pct = np.divide(mat * 100.0, row_totals, out=np.zeros_like(mat, dtype=float), where=row_totals != 0)
annot = np.array([[f'{mat_pct[r,c]:.1f}%\nn={mat[r,c]:,}' for c in range(2)] for r in range(2)])
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(mat_pct, annot=annot, fmt='', cmap='Blues', vmin=0, vmax=100,
            xticklabels=class_names, yticklabels=class_names, cbar=True,
            linewidths=0.8, linecolor='white', annot_kws={'size': 11}, ax=ax)
ax.set_title(f'Confusion Matrix ? {selected_model_name} (test)')
ax.set_xlabel('Prediksi model'); ax.set_ylabel('Kelas aktual')
plt.tight_layout()
plt.savefig(OUT / 'nb07_selected_confusion_matrix.png', bbox_inches='tight', dpi=160)
plt.show()


## 5. Precision, Recall, dan F1 per Kelas


In [ ]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(y_test, pred_test, labels=[0, 1], zero_division=0)
per_class = pd.DataFrame({'Kelas': class_names, 'Precision': precision, 'Recall': recall, 'F1': f1, 'Support': support})
display(per_class.style.format({'Precision':'{:.4f}', 'Recall':'{:.4f}', 'F1':'{:.4f}'}))


## 6. Feature Importance Kandidat Terkunci


In [ ]:
if hasattr(selected_model, 'feature_importances'):
    importances = np.asarray(selected_model.feature_importances(), dtype=float)
    if importances.size == len(feature_cols):
        top_idx = np.argsort(importances)[::-1][:20]
        top_names = [feature_cols[i] for i in top_idx]
        top_values = importances[top_idx]
        fig, ax = plt.subplots(figsize=(9, 7))
        ax.barh(top_names[::-1], top_values[::-1], color='#4472C4')
        ax.set_title(f'Top 20 Feature Importance ? {selected_model_name}')
        ax.set_xlabel('Importance')
        plt.tight_layout()
        plt.savefig(OUT / 'nb07_selected_feature_importance.png', bbox_inches='tight')
        plt.show()
    else:
        print('Panjang feature importance tidak cocok dengan skema fitur; bagian dilewati.')
else:
    print('Model terpilih tidak menyediakan feature_importances(); bagian dilewati.')


## 7. Interpretasi SHAP


In [ ]:
try:
    import shap
    shap_clf = getattr(selected_model, 'clf', None)
    if shap_clf is None:
        print('Objek classifier internal tidak tersedia; SHAP dilewati.')
    else:
        X_shap = X_test[:min(300, len(X_test))]
        explainer = shap.TreeExplainer(shap_clf)
        shap_values = explainer.shap_values(X_shap)
        if isinstance(shap_values, list):
            sv = np.asarray(shap_values[1])
        else:
            sv = np.asarray(shap_values)
            if sv.ndim == 3:
                sv = sv[:, :, 1] if sv.shape[0] == len(X_shap) else sv[1]
        shap.summary_plot(sv, X_shap, feature_names=feature_cols, show=False, max_display=20)
        plt.title(f'SHAP ? {selected_model_name} (test sample)')
        plt.tight_layout()
        plt.savefig(OUT / 'nb07_selected_shap.png', bbox_inches='tight')
        plt.show()
except ImportError:
    print('SHAP tidak terpasang; interpretasi SHAP dilewati.')


## Ringkasan dan Selesai


In [ ]:
final_summary = {
    'selection_metric': 'average_precision on validation',
    'selected_candidate': selected_val_name,
    'selected_model_artifact': selected_model_file,
    'validation_average_precision': selected_val_ap,
    'decision_threshold': selected_threshold if selected_threshold is not None else 0.5,
    'test_metrics': {name: float(final_result[name]) for name in metric_names if name in final_result},
    'test_rows': int(len(y_test)),
    'test_positive_rate': float(y_test.mean()),
}
summary_path = OUT / 'nb07_selected_candidate_summary.json'
summary_path.write_text(json.dumps(final_summary, indent=2, ensure_ascii=False), encoding='utf-8')
print(f"Kandidat terkunci: {selected_val_name} (AP validation {selected_val_ap:.4f})")
print('Test set hanya mengestimasi kandidat terkunci; hasilnya tidak mengubah pemilihan model.')
print(f'Ringkasan: {summary_path}')
